# Evaluation of Fusion Algorithms

Miguel Angel Ramírez

22.01.2025

This notebook aims to evaluate orthoimages for the function experiments between PlanetScope and Sentinel-2.

This notebook:
- Read the bands and calculate the quality indicators.
- Stack the images for the next step of classification.

In [2]:
import numpy as np
import os
import rasterio
import os
import matplotlib.pyplot as plt
import pandas as pd
import csv

from osgeo import gdal, gdalconst
from rasterio import Affine
from rasterio.enums import Resampling
from rasterio.merge import merge

In [ ]:
def load_image(image_path):
    """
    Carga una imagen de una banda desde la ruta especificada.

    Args:
        image_path (str): Ruta del archivo raster.

    Returns:
        numpy.ndarray: Banda leída como arreglo float64.
    """
    with rasterio.open(image_path) as ds:
        return ds.read(1).astype(np.float64)


def _prepare_valid_pixels(original_image, recovered_image):
    """
    Verifica dimensiones y filtra píxeles válidos (finitos en ambas imágenes).

    Args:
        original_image (np.ndarray): Imagen de referencia.
        recovered_image (np.ndarray): Imagen fusionada.

    Returns:
        tuple[np.ndarray, np.ndarray]: Arreglos 1D filtrados.
    """
    if original_image.shape != recovered_image.shape:
        raise ValueError("Las imágenes no tienen la misma dimensión.")

    mask = np.isfinite(original_image) & np.isfinite(recovered_image)
    o = original_image[mask].ravel()
    f = recovered_image[mask].ravel()

    if o.size == 0:
        raise ValueError("No hay píxeles válidos para calcular métricas.")

    return o, f


def calculate_rmse(original_image, recovered_image):
    """
    Calcula el Root Mean Square Error (RMSE).

    Fórmula:
        RMSE = sqrt( (1/n) * sum((O_i - F_i)^2) )
    """
    o, f = _prepare_valid_pixels(original_image, recovered_image)
    return np.sqrt(np.mean((o - f) ** 2))


def calculate_cc(original_image, recovered_image):
    """
    Calcula el Correlation Coefficient (CC) tipo Pearson.

    Fórmula:
        CC = cov(O, F) / (sigma_O * sigma_F)
    """
    o, f = _prepare_valid_pixels(original_image, recovered_image)

    o_mean = np.mean(o)
    f_mean = np.mean(f)

    covar = np.mean((o - o_mean) * (f - f_mean))
    o_std = np.std(o)
    f_std = np.std(f)

    # Evitar división por cero
    if o_std == 0 or f_std == 0:
        return np.nan

    return covar / (o_std * f_std)


def calculate_ergas(original_image, recovered_image, h=10.0, l=3.0):
    """
    Calcula ERGAS en forma monobanda.

    Para una sola banda:
        ERGAS = 100 * (h / l) * (RMSE / mu_O)

    donde:
        h = resolución espacial de la imagen de baja resolución
        l = resolución espacial de la imagen de alta resolución
        mu_O = media de la imagen de referencia

    Args:
        original_image (np.ndarray): Imagen de referencia.
        recovered_image (np.ndarray): Imagen fusionada.
        h (float): Resolución de la imagen de baja resolución.
        l (float): Resolución de la imagen de alta resolución.

    Returns:
        float: Valor ERGAS.
    """
    o, f = _prepare_valid_pixels(original_image, recovered_image)

    rmse = np.sqrt(np.mean((o - f) ** 2))
    mu_o = np.mean(o)

    if mu_o == 0:
        return np.nan

    return 100.0 * (h / l) * (rmse / mu_o)


def calculate_uqi(original_image, recovered_image):
    """
    Calcula el Universal Image Quality Index (UQI) de Wang & Bovik (2002).

    Fórmula:
        UQI = [4 * mu_O * mu_F * sigma_OF] /
              [ (mu_O^2 + mu_F^2) * (sigma_O^2 + sigma_F^2) ]
    """
    o, f = _prepare_valid_pixels(original_image, recovered_image)

    mu_o = np.mean(o)
    mu_f = np.mean(f)

    var_o = np.var(o)
    var_f = np.var(f)
    cov_of = np.mean((o - mu_o) * (f - mu_f))

    denom = (mu_o**2 + mu_f**2) * (var_o + var_f)
    if denom == 0:
        return np.nan

    return (4.0 * mu_o * mu_f * cov_of) / denom


def calculate_metrics_by_band(
    original_folder,
    methods_folder,
    band_numbers,
    h=10.0,
    l=3.0,
    output_file=None
):
    """
    Calcula métricas por banda y método.

    Args:
        original_folder (str): Carpeta con bandas de referencia.
        methods_folder (str): Carpeta con imágenes fusionadas.
        band_numbers (list[int]): Lista de bandas a evaluar.
        h (float): Resolución de la imagen de baja resolución.
        l (float): Resolución de la imagen de alta resolución.
        output_file (str | None): CSV de salida.

    Returns:
        list[dict]: Métricas calculadas.
    """
    band_metrics = []

    for band_num in band_numbers:
        original_path = os.path.join(original_folder, f"band{band_num}.tif")
        original_image = load_image(original_path)

        for method_file in os.listdir(methods_folder):
            if method_file.startswith(f"band{band_num}_") and method_file.endswith(".tif"):
                method_name = os.path.splitext(method_file)[0].replace(f"band{band_num}_", "")
                recovered_path = os.path.join(methods_folder, method_file)
                recovered_image = load_image(recovered_path)

                try:
                    rmse = calculate_rmse(original_image, recovered_image)
                    ergas = calculate_ergas(original_image, recovered_image, h=h, l=l)
                    cc = calculate_cc(original_image, recovered_image)
                    uqi = calculate_uqi(original_image, recovered_image)
                except ValueError:
                    rmse = np.nan
                    ergas = np.nan
                    cc = np.nan
                    uqi = np.nan

                metrics = {
                    "band": band_num,
                    "method": method_name,
                    "rmse": rmse,
                    "ergas": ergas,
                    "cc": cc,
                    "uqi": uqi
                }
                band_metrics.append(metrics)

    if output_file:
        with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
            fieldnames = ["band", "method", "rmse", "ergas", "cc", "uqi"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(band_metrics)

    return band_metrics

# Quality Fusion Indicators

In [ ]:
original_folder = "TOLIMA 2023/S2_bands"
methods_folder = "TOLIMA 2023/S2_f"
band_numbers = [2, 3, 4, 5, 6, 7, 8, 10, 11, 12]
scaling_factor = 1
c = 1

metrics = calculate_metrics_by_band(original_folder, methods_folder, band_numbers, scaling_factor, c,output_file= "TOLIMA 2023/MONTES 2023.csv")

# Graphics

In [ ]:
# Assuming you have the 'band_metrics' dictionary result

# Convert the dictionary to a pandas DataFrame for easier plotting
df = pd.DataFrame(metrics)

# Create subplots for each band
for band_num in df['band'].unique():
    df_band = df[df['band'] == band_num]

    # Create a figure for the current band
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot performance metrics for each method
    for method in df_band['method'].unique():
        df_method = df_band[df_band['method'] == method]
        ax.plot(df_method['method'], df_method['rmse'], label=method)
    
    # Customize the plot
    ax.set_title(f'Band {band_num} Performance')
    ax.set_xlabel('Methods')
    ax.set_ylabel('RMSE')
    ax.legend(loc='best')
    
    # Show or save the plot as needed
    plt.show()  # Show the plot
    # To save the plot as an image, use the following line:
    # plt.savefig(f'band{band_num}_performance.png')


# INDEX ANALYSIS

In [ ]:
# Convert the dictionary to a pandas DataFrame
df = pd.DataFrame(metrics)

# Pivot the DataFrame to create the table
pivot_table = df.pivot(index='band', columns='method')

# Display the pivot table
print(pivot_table)

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

# Load the original "band11" raster
original_band11_path = "path_to_original_band11.tif"
with rasterio.open(original_band11_path) as src:
    original_band11 = src.read(1)  # Assuming it's a single-band raster

# Create a figure
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the original "band11"
show(original_band11, ax=ax, cmap='viridis', title='Original band11')

# Load and plot the processed results for "band11"
methods_folder = "path_to_processed_results_folder"
method_names = ["method1", "method2", "method3"]  # Update with your method names
for method_name in method_names:
    method_band11_path = f"{methods_folder}/{method_name}_band11_fusion.tif"
    with rasterio.open(method_band11_path) as src:
        method_band11 = src.read(1)  # Assuming it's a single-band raster

    show(method_band11, ax=ax, cmap='viridis', title=f'{method_name} Fusion band11')

# Show the plot
plt.show()


# Get Output Images

In [4]:
# Ejemplo de uso:
fusion_folder = "E:\PLANETSCOPE\MONTES 2011\S2_f"
output_folder = "E:\PLANETSCOPE\MONTES 2011\Final_Image"
#methods = ["rcs", "lmvm", "bayes","avg"]
methods = [ "bayes"]

for method in methods:
    stack_fusion_images(fusion_folder, output_folder, method)

Imagen de stack layer para el método bayes creada en E:\PLANETSCOPE\MONTES 2011\Final_Image\S2_bayes_fusion_stack.tif.
